In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install pytorch-fid

In [3]:
!unzip /content/drive/MyDrive/OMR-Datasets/OMR_5Fold_ROIs_split_v3.zip

Kết quả truyền trực tuyến bị cắt bớt đến 5000 dòng cuối.
  inflating: content/OMR_5Fold_ROIs_split/Fold_2/val/confirmed/exam5_50_1_box9.jpg  
  inflating: content/OMR_5Fold_ROIs_split/Fold_2/val/confirmed/exam2_67_1_box24.jpg  
  inflating: content/OMR_5Fold_ROIs_split/Fold_2/val/confirmed/exam4_52_1_box6.jpg  
  inflating: content/OMR_5Fold_ROIs_split/Fold_2/val/confirmed/exam5_83_1_box26.jpg  
  inflating: content/OMR_5Fold_ROIs_split/Fold_2/val/confirmed/exam5_222_1_box15.jpg  
  inflating: content/OMR_5Fold_ROIs_split/Fold_2/val/confirmed/exam5_47_1_box0.jpg  
  inflating: content/OMR_5Fold_ROIs_split/Fold_2/val/confirmed/exam5_24_1_box40.jpg  
  inflating: content/OMR_5Fold_ROIs_split/Fold_2/val/confirmed/exam5_128_1_box29.jpg  
  inflating: content/OMR_5Fold_ROIs_split/Fold_2/val/confirmed/exam5_88_1_box36.jpg  
  inflating: content/OMR_5Fold_ROIs_split/Fold_2/val/confirmed/exam5_190_1_box47.jpg  
  inflating: content/OMR_5Fold_ROIs_split/Fold_2/val/confirmed/exam5_57_1_box19.jpg

In [ ]:
import os
import shutil
import torch
import torch.nn as nn
from PIL import Image, ImageOps
import torchvision.utils as vutils
import subprocess
import re
import glob

# =============================================================================
# 1. CẤU HÌNH ĐƯỜNG DẪN VÀ THAM SỐ
# =============================================================================
BASE_FOLDS_DIR = "/content/content/OMR_5Fold_ROIs_split"
GAN_DIR = "/content/drive/MyDrive/OMR-Datasets/gan"
Z_DIM = 100
IMAGE_SIZE = 64
NUM_FAKE_IMAGES = 1000 # Số ảnh sinh ra để tính FID
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# =============================================================================
# 2. KIẾN TRÚC GENERATOR 
# =============================================================================
class Generator(nn.Module):
    def __init__(self):
        super(Generator, self).__init__()
        self.main = nn.Sequential(
            nn.ConvTranspose2d(Z_DIM, 512, 4, 1, 0, bias=False),
            nn.BatchNorm2d(512), nn.ReLU(True),
            nn.ConvTranspose2d(512, 256, 4, 2, 1, bias=False),
            nn.BatchNorm2d(256), nn.ReLU(True),
            nn.ConvTranspose2d(256, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128), nn.ReLU(True),
            nn.ConvTranspose2d(128, 64, 4, 2, 1, bias=False),
            nn.BatchNorm2d(64), nn.ReLU(True),
            nn.ConvTranspose2d(64, 3, 4, 2, 1, bias=False),
            nn.Tanh()
        )
    def forward(self, input):
        return self.main(input)

# =============================================================================
# 3. CÁC HÀM HỖ TRỢ TIỀN XỬ LÝ VÀ ĐÁNH GIÁ
# =============================================================================
def pil_square_pad(img, fill_color=(255, 255, 255)):
    """Hàm đắp viền trắng tạo ảnh vuông"""
    w, h = img.size
    max_wh = max(w, h)
    left = int((max_wh - w) / 2)
    top = int((max_wh - h) / 2)
    right = max_wh - w - left
    bottom = max_wh - h - top
    return ImageOps.expand(img, (left, top, right, bottom), fill=fill_color)

def generate_fake_images(generator, output_dir, num_images=1000, batch_size=100):
    """Sinh ảnh GAN và lưu vào ổ cứng để chuẩn bị tính FID"""
    if os.path.exists(output_dir):
        shutil.rmtree(output_dir)
    os.makedirs(output_dir)

    generator.eval()
    batches = num_images // batch_size
    remainder = num_images % batch_size
    img_count = 0

    with torch.no_grad():
        for i in range(batches + (1 if remainder > 0 else 0)):
            current_batch = remainder if (i == batches) else batch_size
            noise = torch.randn(current_batch, Z_DIM, 1, 1, device=DEVICE)
            fake = generator(noise)
            # Chuẩn hóa từ [-1, 1] về [0, 1] để lưu ảnh
            fake = (fake + 1) / 2.0

            for j in range(current_batch):
                img_path = os.path.join(output_dir, f"fake_{img_count:04d}.jpg")
                vutils.save_image(fake[j], img_path)
                img_count += 1

def compute_fid(real_dir, fake_dir):
    """Gọi ngầm thư viện pytorch-fid và trích xuất điểm số"""
    try:
        result = subprocess.run(
            ['python', '-m', 'pytorch_fid', real_dir, fake_dir, '--device', str(DEVICE)],
            capture_output=True, text=True, check=True
        )
        # Tìm dòng chứa FID: [Số]
        match = re.search(r'FID:\s+([0-9\.]+)', result.stdout)
        if match:
            return float(match.group(1))
        else:
            print("⚠ Không tìm thấy điểm FID trong log:", result.stdout)
            return float('inf')
    except Exception as e:
        print(f"❌ Lỗi khi tính FID: {e}")
        return float('inf')

# =============================================================================
# 4. CHẠY PIPELINE TÌM EPOCH VÔ ĐỊCH CHO 5 FOLDS
# =============================================================================
def main():
    print("🚀 BẮT ĐẦU QUÁ TRÌNH TÌM GAN TỐT NHẤT CHO 5 FOLD 🚀")

    best_overall_results = []

    for fold in range(1, 6):
        print(f"\n{'='*60}")
        print(f"🔍 ĐANG KIỂM TRA FOLD {fold}")
        print(f"{'='*60}")

        fold_dir = os.path.join(BASE_FOLDS_DIR, f"Fold_{fold}")
        real_data_dir = os.path.join(fold_dir, "train_Scen1_withoutGAN", "crossedout")
        weights_dir = os.path.join(GAN_DIR, "checkpoints", f"Fold_{fold}")

        # Thư mục tạm thời
        real_tmp_dir = os.path.join(fold_dir, "tmp_real_64x64")
        fake_tmp_dir = os.path.join(fold_dir, "tmp_fake_eval")
        # Thư mục lưu 1000 ảnh GAN xịn nhất cuối cùng
        best_gan_output_dir = os.path.join(GAN_DIR, "gan_images_best", f"Fold_{fold}")

        # 1. BƯỚC CHUẨN BỊ ẢNH THẬT (Apples-to-Apples)
        if os.path.exists(real_tmp_dir): shutil.rmtree(real_tmp_dir)
        os.makedirs(real_tmp_dir)

        real_images = glob.glob(os.path.join(real_data_dir, "*.*"))
        print(f"   ⏳ Đang chuẩn bị {len(real_images)} ảnh Thật (Đắp viền + Resize 64x64) để tính FID...")
        for img_path in real_images:
            try:
                img = Image.open(img_path).convert('RGB')
                img_padded = pil_square_pad(img)
                img_resized = img_padded.resize((IMAGE_SIZE, IMAGE_SIZE), Image.Resampling.BICUBIC)
                img_name = os.path.basename(img_path)
                img_resized.save(os.path.join(real_tmp_dir, img_name))
            except: continue

        # 2. QUÉT CHECKPOINTS VÀ TÍNH FID
        weight_files = sorted(glob.glob(os.path.join(weights_dir, "netG_epoch_*.pth")))
        if not weight_files:
            print(f"   ❌ Không tìm thấy file trọng số nào trong {weights_dir}")
            continue

        best_fid = float('inf')
        best_epoch_file = None

        netG = Generator().to(DEVICE)

        print(f"   ⏳ Bắt đầu tính FID cho {len(weight_files)} checkpoints...")
        for w_file in weight_files:
            epoch_name = os.path.basename(w_file)

            # Tải trọng số
            netG.load_state_dict(torch.load(w_file, map_location=DEVICE))

            # Sinh ảnh tạm
            generate_fake_images(netG, fake_tmp_dir, num_images=NUM_FAKE_IMAGES)

            # Tính FID
            current_fid = compute_fid(real_tmp_dir, fake_tmp_dir)
            print(f"      - {epoch_name}: FID = {current_fid:.2f}")

            # Cập nhật kỷ lục
            if current_fid < best_fid:
                best_fid = current_fid
                best_epoch_file = w_file

        print(f"   🏆 WINNER FOLD {fold}: {os.path.basename(best_epoch_file)} (FID = {best_fid:.2f})")
        best_overall_results.append({'Fold': fold, 'Best_Epoch': os.path.basename(best_epoch_file), 'FID': best_fid})

        # 3. SINH ẢNH CHỐT HẠ CHO KỊCH BẢN 2
        print(f"   ⏳ Đang đẻ 1000 ảnh bằng trọng số Vô địch vào mục [gan_images_best]...")
        netG.load_state_dict(torch.load(best_epoch_file, map_location=DEVICE))
        generate_fake_images(netG, best_gan_output_dir, num_images=1000)

        # 4. DỌN DẸP RÁC
        shutil.rmtree(real_tmp_dir)
        shutil.rmtree(fake_tmp_dir)
        print(f"   ✅ Hoàn thành Fold {fold}!")

    # TỔNG KẾT
    print("\n" + "="*50)
    print("🏆 BẢNG VÀNG CHỌN EPOCH GAN TỐT NHẤT 5 FOLDS 🏆")
    print("="*50)
    for res in best_overall_results:
        print(f"Fold {res['Fold']}: Dùng {res['Best_Epoch']} - Đạt FID: {res['FID']:.2f}")
    print("="*50)
    print("Tất cả 5.000 ảnh GAN xịn nhất đã sẵn sàng trong thư mục [gan_images_best] của từng Fold!")

if __name__ == '__main__':
    main()

🚀 BẮT ĐẦU QUÁ TRÌNH TÌM GAN TỐT NHẤT CHO 5 FOLD 🚀

🔍 ĐANG KIỂM TRA FOLD 1
   ⏳ Đang chuẩn bị 1500 ảnh Thật (Đắp viền + Resize 64x64) để tính FID...
   ⏳ Bắt đầu tính FID cho 24 checkpoints...
      - netG_epoch_100.pth: FID = 124.34
      - netG_epoch_110.pth: FID = 117.54
      - netG_epoch_120.pth: FID = 113.08
      - netG_epoch_130.pth: FID = 109.32
      - netG_epoch_140.pth: FID = 105.11
      - netG_epoch_150.pth: FID = 103.58
      - netG_epoch_160.pth: FID = 103.22
      - netG_epoch_170.pth: FID = 106.17
      - netG_epoch_180.pth: FID = 100.30
      - netG_epoch_190.pth: FID = 99.88
      - netG_epoch_200.pth: FID = 102.86
      - netG_epoch_210.pth: FID = 101.15
      - netG_epoch_220.pth: FID = 91.76
      - netG_epoch_230.pth: FID = 93.93
      - netG_epoch_240.pth: FID = 99.58
      - netG_epoch_250.pth: FID = 99.88
      - netG_epoch_260.pth: FID = 95.07
      - netG_epoch_270.pth: FID = 94.00
      - netG_epoch_280.pth: FID = 92.42
      - netG_epoch_290.pth: FID = 92.